# Raport: Metody Systemowe i Decyzyjne

**Autor:** Michał Śliwa

### Charakterystyka cech:

| Cecha | Typ | Opis |
|-------|-----|------|
| `age` | Liczbowa ciągła | Wiek pacjenta (lata) |
| `trestbps` | Liczbowa ciągła | Spoczynkowe ciśnienie krwi (mm Hg) |
| `chol` | Liczbowa ciągła | Cholesterol w surowicy (mg/dl) |
| `thalach` | Liczbowa ciągła | Maksymalne osiągnięte tętno |
| `oldpeak` | Liczbowa ciągła | Obniżenie odcinka ST wywołane wysiłkiem |
| `sex` | Kategorialna | Płeć (1 = mężczyzna, 0 = kobieta) |
| `cp` | Kategorialna | Rodzaj bólu w klatce piersiowej (1–4) |
| `fbs` | Kategorialna | Cukier na czczo > 120 mg/dl (1 = tak) |
| `restecg` | Kategorialna | Wynik EKG w spoczynku (0–2) |
| `exang` | Kategorialna | Dławica wywołana wysiłkiem (1 = tak) |
| `slope` | Kategorialna | Nachylenie odcinka ST (1–3) |
| `ca` | Kategorialna | Liczba głównych naczyń wieńcowych (0–3) |
| `thal` | Kategorialna | Wynik testu talu (3, 6, 7) |
| `target` | Zmienna docelowa | Diagnoza: 0 = zdrowy, 1 = chory |

# 1. Przygotowanie danych
Na początku przygotowuję dane potrzebne do pracy podczas drzewa decyzyjnego. Dzielę wczytane dane na zbiór treningowy oraz testowy w proporcji 4:1.

In [2]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import accuracy_score, precision_recall_fscore_support, mean_squared_error

# Wczytanie danych
columns = ['age', 'sex', 'cp', 'trestbps', 'chol', 'fbs', 'restecg', 'thalach', 'exang', 'oldpeak', 'slope', 'ca', 'thal', 'target']
df = pd.read_csv('data/processed.cleveland.data', header=None, names=columns, na_values='?')
df = df.dropna()
df['target_bin'] = (df['target'] > 0).astype(int) # Klasyfikacja (0/1)

# Podział na zbiory (Data Leakage prevention)
X = df.drop(['target', 'target_bin'], axis=1)
y_cls = df['target_bin'] # Cel dla klasyfikacji
y_reg = df['thalach']   # Cel dla regresji (np. tętno)

X_train, X_test, y_train_cls, y_test_cls = train_test_split(X, y_cls, test_size=0.2, random_state=42)
X_train_reg, X_test_reg, y_train_reg, y_test_reg = train_test_split(X, y_reg, test_size=0.2, random_state=42)

# 2. Tworzenie drzewa decyzyjnego
Następnie tworzymy drzewo decyzyjne oraz metrykę przy użyciu biblioteki scikit-learn.

In [3]:
from sklearn.tree import DecisionTreeClassifier

# Trening modelu
tree_model = DecisionTreeClassifier(random_state=42)
tree_model.fit(X_train, y_train_cls)

# Predykcja i metryki dla niezbalansowanych danych
y_pred_cls = tree_model.predict(X_test)
precision, recall, f1, _ = precision_recall_fscore_support(y_test_cls, y_pred_cls, average='binary')

print(f"Metryki na zbiorze testowym:")
print(f"Accuracy: {accuracy_score(y_test_cls, y_pred_cls):.2f}")
print(f"F1 Score: {f1:.2f}")
print(f"Precision: {precision:.2f}")
print(f"Recall: {recall:.2f}")

Metryki na zbiorze testowym:
Accuracy: 0.78
F1 Score: 0.75
Precision: 0.69
Recall: 0.83


**Accuracy** - Dokładność

Dla ilu przypadków model wskazał poprawnie chorobę u chorego, i brak choroby u zdrowego

**Precision** - Precyzja

Dla ilu przypadków model wskazał chorobę faktycznie u osoby chorej

**Recall** - Czułość

Dla ilu przypadków model wskazał poprawnie chorobę u chorego, pomijając niektóre osoby i kwalifikując je jako zdrowe

**F1 Score**

Jakość modelu, jako średnia harmoniczna

# 3. Własna entropia i interpretacja
Implementujemy od zera funkcje w NumPy

In [4]:
def entropy(y):
    probs = y.value_counts(normalize=True)
    return -np.sum(probs * np.log2(probs + 1e-9))

def information_gain(y, y_left, y_right):
    parent_entropy = entropy(y)
    n = len(y)
    n_l, n_r = len(y_left), len(y_right)
    child_entropy = (n_l/n) * entropy(y_left) + (n_r/n) * entropy(y_right)
    return parent_entropy - child_entropy

# Przykład wizualizacji reguł dla płytkiego drzewa
short_tree = DecisionTreeClassifier(max_depth=3, random_state=42)
short_tree.fit(X_train, y_train_cls)

from sklearn.tree import export_text
print("Zasady wygenerowane przez algorytm (max_depth=3):")
print(export_text(short_tree, feature_names=list(X.columns)))

Zasady wygenerowane przez algorytm (max_depth=3):
|--- ca <= 0.50
|   |--- thal <= 6.50
|   |   |--- oldpeak <= 2.70
|   |   |   |--- class: 0
|   |   |--- oldpeak >  2.70
|   |   |   |--- class: 1
|   |--- thal >  6.50
|   |   |--- cp <= 3.50
|   |   |   |--- class: 0
|   |   |--- cp >  3.50
|   |   |   |--- class: 1
|--- ca >  0.50
|   |--- cp <= 3.50
|   |   |--- oldpeak <= 0.35
|   |   |   |--- class: 0
|   |   |--- oldpeak >  0.35
|   |   |   |--- class: 1
|   |--- cp >  3.50
|   |   |--- trestbps <= 109.00
|   |   |   |--- class: 0
|   |   |--- trestbps >  109.00
|   |   |   |--- class: 1



## Reguły z poprzedniej listy

| # | Reguła | Uzasadnienie                                     |
|---|--------|--------------------------------------------------|
| 1 | `ca > 0` | Zablokowane naczynia = chory                     |
| 2 | `exang == 1` | Ból przy wysiłku = chory                         |
| 3 | `cp == 4` | Brak bólu paradoksalnie = chory                  |
| 4 | `age > 55 AND thalach < 140` | Starsi pacjenci z niskim tętnem maks. = chory    |
| 5 | `sex == 1 AND age > 60 AND chol > 260` | Starsi mężczyźni z wysokim cholesterolem = chory |

## Aktualne reguły

| # | Reguła | Uzasadnienie                                                              |
|---|--------|---------------------------------------------------------------------------|
| 1 | `ca <= 0.50 AND thal <= 6.50 AND oldpeak <= 2.70` | Czyste naczynia, dobry test i małe obniżenie ST = zdrowy                  |
| 2 | `ca <= 0.50 AND thal <= 6.50 AND oldpeak > 2.70` | Prawidłowe naczynia i test, ale duże obniżenie ST pod obciążeniem = chory |
| 3 | `ca <= 0.50 AND thal > 6.50 AND cp > 3.50` | Brak zablokowanych naczyń, ale zły test talowy i nietypowy ból = chory    |
| 4 | `ca > 0.50 AND cp <= 3.50 AND oldpeak <= 0.35` | Zablokowane naczynia i ból dławicowy, ale brak obniżenia ST = zdrowy      |
| 5 | `ca > 0.50 AND cp > 3.50 AND trestbps > 109.00` | Zablokowane naczynia, "cichy" ból i ciśnienie wyższe niż 109 = chory      |

In [5]:
print("\n--- Dowód na Zysk Informacyjny (Information Gain) ---")
# 1. Wybieramy najważniejszą cechę z drzewa (np. thal, próg z drzewa to zazwyczaj < 4.5, my mamy kategorialne 3,6,7, więc np. thal == 3)
y_left_good = y_train_cls[X_train['thal'] == 3.0]
y_right_good = y_train_cls[X_train['thal'] != 3.0]
ig_good = information_gain(y_train_cls, y_left_good, y_right_good)

# 2. Wybieramy najmniej ważną cechę (np. fbs - cukier na czczo)
y_left_bad = y_train_cls[X_train['fbs'] == 0]
y_right_bad = y_train_cls[X_train['fbs'] == 1]
ig_bad = information_gain(y_train_cls, y_left_bad, y_right_bad)

print(f"Zysk informacyjny dla cechy silnej (thal): {ig_good:.4f}")
print(f"Zysk informacyjny dla cechy słabej (fbs): {ig_bad:.4f}")
print("Wniosek: Cecha 'thal' daje znacznie większy spadek entropii (zanieczyszczenia) niż 'fbs'.")


--- Dowód na Zysk Informacyjny (Information Gain) ---
Zysk informacyjny dla cechy silnej (thal): 0.1822
Zysk informacyjny dla cechy słabej (fbs): 0.0017
Wniosek: Cecha 'thal' daje znacznie większy spadek entropii (zanieczyszczenia) niż 'fbs'.


# 4. Regresja liniowa
Implementacja klasy z dwiema metodami trenowania: analityczną i gradientową.

In [10]:
from sklearn.linear_model import LinearRegression
import numpy as np

class MyLinearRegression:
    def __init__(self):
        self.weights = None

    def fit_analytical(self, X, y):
        X_b = np.c_[np.ones((len(X), 1)), X]
        # Metoda najmniejszych kwadratów (macierz pseudoodwrotna)
        self.weights = np.linalg.pinv(X_b.T @ X_b) @ X_b.T @ y

    def fit_gradient(self, X, y, lr=0.01, epochs=1000):
        X_b = np.c_[np.ones((len(X), 1)), X]
        self.weights = np.zeros(X_b.shape[1])
        for _ in range(epochs):
            gradients = 2/len(X) * X_b.T @ (X_b @ self.weights - y)
            self.weights -= lr * gradients

    def predict(self, X):
        X_b = np.c_[np.ones((len(X), 1)), X]
        return X_b @ self.weights

# --- Realizacja wymogu 4.0: Porównanie 3 metod trenowania ---

# 1. Nasza implementacja - Metoda Analityczna
my_model_ana = MyLinearRegression()
my_model_ana.fit_analytical(X_train_reg, y_train_reg)

# 2. Nasza implementacja - Spadek Gradientu (baaaardzo mały krok, żeby nie wywaliło 'inf' na dużych liczbach)
my_model_grad = MyLinearRegression()
my_model_grad.fit_gradient(X_train_reg, y_train_reg, lr=1e-7, epochs=50000)

# 3. Model scikit-learn
sklearn_model = LinearRegression()
sklearn_model.fit(X_train_reg, y_train_reg)

# Wyświetlamy pierwsze 5 wag (indeks 0 w naszych modelach to 'intercept', czyli przesunięcie, więc pomijamy go dla zgodności z coef_)
print("Porównanie wyuczonych wag (pierwsze 5 cech):")
print(f"Scikit-learn:      {np.round(sklearn_model.coef_[:5], 4)}")
print(f"Nasz Analityczny:  {np.round(my_model_ana.weights[1:6], 4)}")
print(f"Nasz Gradientowy:  {np.round(my_model_grad.weights[1:6], 4)}")

# Liczymy MSE z naszego najlepszego analitycznego modelu
y_pred_reg = my_model_ana.predict(X_test_reg)
print(f"\nMSE (Nasza metoda analityczna na zbiorze testowym): {mean_squared_error(y_test_reg, y_pred_reg):.2f}")

Porównanie wyuczonych wag (pierwsze 5 cech):
Scikit-learn:      [-0.8625  3.7916 -2.8866  0.1352  0.0544]
Nasz Analityczny:  [-0.8625  3.7916 -2.8866  0.1352  0.0544]
Nasz Gradientowy:  [0.0349 0.0317 0.0088 0.7736 0.1708]

MSE (Nasza metoda analityczna na zbiorze testowym): 300.41


# 5. Skalowanie i Analiza wag
Tutaj porównuję wpływy cech.

In [11]:
from sklearn.preprocessing import StandardScaler
import pandas as pd
from IPython.display import display

# --- Realizacja wymogu 4.5: Skalowanie i Analiza Wag ---

# 1. Zapisujemy wagi sprzed skalowania (z naszej metody analitycznej)
wagi_przed = pd.Series(my_model_ana.weights[1:], index=X_train_reg.columns)

# 2. Skalujemy dane
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train_reg)
X_test_scaled = scaler.transform(X_test_reg)

# 3. Trenujemy nową instancję naszego modelu na przeskalowanych danych
model_po = MyLinearRegression()
model_po.fit_analytical(X_train_scaled, y_train_reg)
wagi_po = pd.Series(model_po.weights[1:], index=X_train_reg.columns)

# 4. Zestawienie i wyświetlenie wag
porownanie_wag = pd.DataFrame({
    'Waga (Przed Skalowaniem)': wagi_przed,
    'Waga (Po Skalowaniu)': wagi_po
}).sort_values(by='Waga (Po Skalowaniu)', key=abs, ascending=False) # Sortujemy po wartości bezwzględnej

print("Porównanie wag Regresji Liniowej dla przewidywania zmiennej 'thalach':\n")
display(porownanie_wag.round(3))

Porównanie wag Regresji Liniowej dla przewidywania zmiennej 'thalach':



,Waga (Przed Skalowaniem),Waga (Po Skalowaniu)
age,-0.863,-7.731
slope,-7.895,-4.813
target_bin,-7.941,-3.966
exang,-7.234,-3.451
chol,0.054,2.845
cp,-2.887,-2.771
trestbps,0.135,2.434
sex,3.792,1.737
oldpeak,-1.259,-1.428
thal,-0.422,-0.821


# Analiza Regresji Liniowej i Skalowania

**1. Znaczenie i interpretacja wag:**
W powyższym modelu regresji liniowej (przewidującym maksymalne tętno `thalach`), waga przy danej cesze mówi nam, jak bardzo dany czynnik wpływa na wynik.
Przykładowo: Jeśli waga przy zmiennej `age` (wiek) wynosi np. **-0.9**, oznacza to, że statystycznie **wzrost wieku pacjenta o 1 rok powoduje spadek przewidywanego maksymalnego tętna o 0.9 uderzenia na minutę**, zakładając, że pozostałe parametry pacjenta pozostają bez zmian.

**2. Dlaczego skalowanie wpłynęło na interpretację ważności cech?**
Gdy spojrzymy na kolumnę "Wagi (Bez skalowania)", interpretacja "najważniejszej" cechy może być myląca. Wynika to z faktu, że cechy operują w drastycznie różnych rzędach wielkości (np. `chol` to setki mg/dl, a `oldpeak` to ułamki). Model może przypisać ogromną wagę ułamkową do zmiennej `oldpeak`, która po pomnożeniu przez małą wartość da ostatecznie nieduży efekt.

Dopiero po zastosowaniu `StandardScaler` (który sprowadza wszystkie cechy do rozkładu o średniej 0 i odchyleniu 1), kolumna **"Wagi (Po skalowaniu)" pozwala na obiektywne porównanie**. Tabela została posortowana po wartościach bezwzględnych przeskalowanych wag, co jasno pokazuje nam, która cecha "najmocniej ciągnie" wynik tętna w dół lub w górę. Bez skalowania porównywalibyśmy przysłowiowe jabłka do pomarańczy.